<a href="https://colab.research.google.com/github/BahruzHuseynov/Portfolio/blob/main/DND_EXAM_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Final Exam for Deep Network Development course**
Name: Bahruz Huseynov

Neptun ID: F0MMI9

Date: 27/05/2024

Duration: 9AM-11AM

## General rules
This notebook contains the task to be solved in order to pass the exam and complete the course.
It contains a task similar to what you have worked on during the semester, which consists on implementing a network architecture and a function. There are additional requirements which are optional.

The exam has a duration of 2 hours. You are free to distribute the time as you please between the different requirements.
During the exam you can use any resource (internet, AI, practice notebooks, etc). **However it is strictly prohibited to use any communication channel** (Teams, WhatsApp, Messenger, etc.). Using any of those will result in immediate **FAIL**.

Your solution should be submitted to Canvas as a .ipynb file!

Please note that, to **PASS** the exam you must **SUBMIT A SUCCESSFUL SOLUTION SATISFYING THE MINIMUM REQUIREMENTS**. If you **FAIL** the exam, you have the right to retry it **ONE MORE TIME**. If you **FAIL AGAIN**, then unfortunately, you have failed the course.

If you **PASS** the exam, then the final grade is the weighted average of your asignment defenses (theory and code).

## Task description & Requirements
The task is to implement a custom architecture and its forward function.
The task is inspired on Image Captioning and the architecture is a U-Net like model with an LSTM at the end. The model receives an image as input and generates text as output (actually a probability distribution over a set of tokens). For the extra part, you are required to combine different inputs and to extend the architecture.

## Requirements:
------------------------------------------------------------------------
**Minimum requirements - ENOUGH TO PASS THE EXAM**
1.   Implement the layers of the architecture shown in section 1. Fill out the unknown parts in order to complete the architecture.
2.   Implement the forward function of the architecture. Make sure that the input and output are correct.

**!!! To complete the requirements 1 and 2, your final output should match the expected output indicated on cell 1.2. !!!**

------------------------------------------------------------------------

**Extra requirements - for grade improvement and potentially access to AI Lab**
3.   Use the output embedding of the encoder implemented on cell 1.1 and fuse it with another provided embedding. --> **+1 in final grade**
4.   Modify the architecture previously implemented to accept the new fused embedding and extend the architecture with the details shown on cell 1.4. --> **Access to AI Lab**
------------------------------------------------------------------------

In [1]:
#Necessary imports
import torch
import torch.nn as nn
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## 1. Architecture

Please right click the image and "Open image in a new tab" to view it better with zoom. Or download it from here: https://drive.google.com/file/d/1P3q6dNxywzIpkmOhxhqN01I5NVlQWW7_/view?usp=sharing

<br>
<br>

![](https://drive.google.com/uc?export=view&id=1P3q6dNxywzIpkmOhxhqN01I5NVlQWW7_)

### 1.1. Implement the architecture and its forward function

In [115]:
t = torch.randn(1, 3, 256, 256)

In [123]:
class CustomNet(nn.Module):
    def __init__(self):
        super(CustomNet, self).__init__()
        #Encoder parts
        self.e1 = nn.Conv2d(3, 16, kernel_size = 1, stride = 2, padding = 0) # 1, 16, 128, 128
        self.e1_1 = nn.MaxPool2d(kernel_size = 2, stride = 2) # 1, 16, 64, 64
        self.e1_2 = nn.Conv2d(16, 16, 3, stride = 2, padding = 1) # ? - 1, 16, 64, 64

        self.e2 = nn.Conv2d(16, 32, kernel_size = 3, stride = 1, padding = 1) # 1, 32, 64, 64
        self.e2_1 = nn.ReLU() # 1, 32, 64, 64
        self.e2_2 = nn.Conv2d(32, 32, kernel_size = 3, stride = 1, padding = 1) # ? - 1, 32, 64, 64

        self.finish_encoder = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size = 5, stride = 2, padding = 0),
            nn.BatchNorm2d(num_features = 128)
        )

        # Decoder parts
        self.decoder_start = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size = 3, stride = 1, padding = 0),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 64, kernel_size = 3, stride = 2, padding = 1)
        ) # 1, 64, 127, 127

        self.d_group1 = nn.Sequential(
            nn.ConvTranspose2d(64, 3, kernel_size = 4, stride = 2, padding = 0),
            nn.Conv2d(3, 16, kernel_size = 11, stride = 8, padding = 0)
        )

        self.lstm_layer = nn.LSTM(16, 64, num_layers = 1, batch_first = True)
        self.final_activation = nn.Softmax2d()

    def forward(self, x):
        print("Input:", x.shape)

        # Define the encoder part
        print("Encoder")
        x = self.e1_2(self.e1(x)) + self.e1_1(self.e1(x)) # Group 1
        group2 = torch.cat((self.e2_1(self.e2(x)), self.e2_2(self.e2(x))), dim = 1) # Group 2 - ([1, 64, 64, 64])
        group2 = nn.functional.interpolate(group2, size=(127, 127), mode='bilinear', align_corners=False)
        print(group2.shape)
        encoder_end = self.finish_encoder(group2) # [1, 128, 62, 62]


        #Define the decoder part
        print("Decoder")
        d1 = self.decoder_start(encoder_end) + group2
        d1 = self.d_group1(d1)
        d1 = d1.reshape(1, -1, 16)

        out, (h_n, c_n) = self.lstm_layer(d1)

        # PLEASE NAME YOUR FINAL OUTPUT AS 'out' OR RENAME THE VARIABLE IN THE RETURN

        return self.final_activation(out)

In [124]:
CustomNet()(t).shape

Input: torch.Size([1, 3, 256, 256])
Encoder
torch.Size([1, 64, 127, 127])
Decoder


torch.Size([1, 961, 64])

### 1.2 Check if your implementation is correct
For a given arbitraty input of size (1,3,256,256) the expected output is (1,961,64)

In [125]:
image = torch.tensor(np.random.rand(3,256,256), dtype=torch.float32)
image = torch.unsqueeze(image, dim=0)

model = CustomNet()
output = model(image)
print()

try:
    assert output.shape == torch.Size([1, 961, 64])
    print("CONGRATULATIONS! You have PASSed the exam by successfully completing the minimum requirements!")
except AssertionError:
    print("Unfortunately, you have FAILed the exam by not being able to complete the minimum requirements.")

Input: torch.Size([1, 3, 256, 256])
Encoder
torch.Size([1, 64, 127, 127])
Decoder

CONGRATULATIONS! You have PASSed the exam by successfully completing the minimum requirements!


# EXTRA REQUIREMENTS (OPTIONAL)

### 1.3. Fuse the embeddings
First obtain the image embeddings from the last layer of the encoder from the previously implemented architecture. Then combine it with the new provided embedding.

Please right click the image and "Open image in a new tab" to view it better with zoom. Or download it from here: https://drive.google.com/file/d/1pts-Dzka5fYD6clW3Pb1NCgEbPWZKF74/view?usp=sharing

<br>
<br>

![](https://drive.google.com/uc?export=view&id=1pts-Dzka5fYD6clW3Pb1NCgEbPWZKF74)

In [ ]:
class CombineEmbeddings(nn.Module):
    def __init__(self, embed_dim, comb_dim):
        super(CombineEmbeddings, self).__init__()
        self.embed_dim = embed_dim
        self.comb_dim = comb_dim

    def forward(self, embedding_1, embedding_2):
        #Reshaping might be needed

        return attention_output

In [ ]:
#REPLACE THIS WITH ACTUAL EMBEDDING FROM ENCODER OUTPUT OF PREVIOUSLY IMPLEMENTED ARCHITECTURE.
#FOR SIMPLICITY REMOVE BATCH SIZE - squeeze
encoder_embedding = torch.randn(128, 62, 62) #REPLACE THIS!!!!

#NEW EMBEDDING
new_embedding = torch.randn(128, 62, 62)

#combine embeddings according to the figure in 1.3.
combine = CombineEmbeddings(embed_dim=128, comb_dim=128) #for simplicity use the same size for the combination 128
output = combine(encoder_embedding, new_embedding)
print()

try:
    assert output.shape == torch.Size([128, 62, 62])
    print("CONGRATULATIONS! You have earned +1 in your final grade by successfully satisfying one of the extra requirements!")
except AssertionError:
    print("Sorry! Keep trying!")

### 1.4. Modify and extend the architecture
Remove all the existing decoder layers and use just a Transformer based decoder. The input should be the new combined embedding.

In [ ]:
print("If you have completed this part, then CONGRATULATIONS! You will be suggested as a potential student to join the AI Lab!")

If you have completed this part, then CONGRATULATIONS! You will be suggested as a potential student to join the AI Lab!
